In [1]:
# 什么是State

"""

state 使得不同工具可以读取或修改同一份数据

"""

from rich.console import Console

console = Console()

state = {

    "papers":[],
    "selected_ids": []

}

# 模拟搜索工具执行
state["papers"] = [
    {"id": "W1", "title": "Paper A", "citations": 100},
    {"id": "W2", "title": "Paper B", "citations": 50}
]

# 模拟排序工具执行
state["selected_ids"] = ["W1"]

console.print("搜索结果:", state["papers"])
console.print("选中论文:", state["selected_ids"])



搜索结果:
[{'id': 'W1', 'title': 'Paper A', 'citations': 100}, {'id': 'W2', 'title': 'Paper B', 'citations': 50}]

选中论文:
['W1']

### 如何让多个Tool 共享 State

In [2]:

# 两个 Tool 共享同一个 State

state = {"papers": [], "selected": []}


def search_papers():
    papers = [
        {"title": "Paper A", "citations": 50},
        {"title": "Paper B", "citations": 120},
        {"title": "Paper C", "citations": 80},
    ]

    # 将搜索结果保存到 State
    state["papers"] = papers

    return {"count": len(papers)}


def rank_papers(top_k: int = 2):
    # 直接从 State 中读取论文
    papers = state["papers"]

    ranked = sorted(
        papers,
        key=lambda p: p["citations"],
        reverse=True
    )[:top_k]

    state["selected"] = ranked

    return ranked


# 模拟 Agent 先后调用两个 Tool
print(search_papers())
print(rank_papers(top_k=2))


{'count': 3}
[{'title': 'Paper B', 'citations': 120}, {'title': 'Paper C', 'citations': 80}]


### 知识点 3：为什么要用 dataclass 管理 State

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AgentState:
    """
    
    这里需要理解两个语法。

    @dataclass：自动为类生成初始化方法等常用方法，让我们专注于定义数据结构。

    field(default_factory=list)：为每个 AgentState 实例创建独立的空列表，而不是多个实例共享同一个默认列表。
    
    """
    papers: list = field(default_factory=list)
    selected_ids: list = field(default_factory=list)
    last_keyword:str = ""

    def snapshot(self):

        return {
            "paper_count": len(self.papers),
            "selected_ids": self.selected_ids
        }

state = AgentState()

state.papers.append("Paper A")
state.selected_ids.append("W1")
state.last_keyword = "mechine learning"

console.print(state)
console.print(state.last_keyword)


AgentState(papers=['Paper A'], selected_ids=['W1'], last_keyword='mechine learning')

mechine learning

### 知识点 4：State 和 Context 有什么区别？

In [4]:
import json

"""

State:是 python 保存的全部数据

context:是实际传给模型的消息和信息

"""

# 模拟保存 100 篇文章
state = {
    "papers":[
        {
            "id": f"W{i}",
            "title": f"Paper {i}",
            "abstract": "Very long abstract" * 100
        }
        for i in range(100)
    ]
}

# State 中保存完整数据
full_data = json.dumps(state)

# 只取三篇论文的ID和标题给模型
context = {
    "paper_count": len(state["papers"]),
    "preview": [
        {"id": p["id"], "title": p["title"]}
        for p in state["papers"][:3]
    ]
}

context_data = json.dumps(context)

console.print("完整 State 字符数:", len(full_data))
console.print("简化 Context 字符数:", len(context_data))
console.print(context_data)





完整 State 字符数: 185192

简化 Context 字符数: 135

{"paper_count": 100, "preview": [{"id": "W0", "title": "Paper 0"}, {"id": "W1", "title": "Paper 1"}, {"id": "W2", 
"title": "Paper 2"}]}

### 六、知识点 5：State 与 Memory 有什么关系？

In [6]:
class MiniAgent:
    def __init__(self):
        self.state = {"selected_paper": None}

    def run(self, query: str):
        if query.startswith("选择"):
            self.state["selected_paper"] = query[2:]
            return "论文已选择"

        if "刚才" in query:
            return f"刚才选择的是：{self.state['selected_paper']}"

        return "没有相关操作"

agent = MiniAgent()

console.print(agent.run("选择Paper A"))
console.print(agent.run("刚才选择了什么论文?"))



论文已选择

刚才选择的是：Paper A